### 1. Import Dependencies

In [12]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
dependency_location = os.getenv("DEPENDENCY_LOCATION")
feature_engineered_location = os.getenv("FEATURE_ENGINEERED_LOCATION")
data_processed_location = os.getenv("DATA_PROCESSED_LOCATION")
models_location = os.getenv("MODELS_LOCATION")

In [ ]:
import joblib
import warnings 
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold, cross_validate
)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)
from sklearn.model_selection import GridSearchCV
warnings.filterwarnings('ignore')

### 2. Load the Data

In [14]:
X_train = pd.read_csv(os.path.join(data_processed_location, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(data_processed_location, "X_test.csv"))
y_train = pd.read_csv(os.path.join(data_processed_location, "y_train.csv")).squeeze()
y_test  = pd.read_csv(os.path.join(data_processed_location, "y_test.csv")).squeeze()

### 2. Define Tuning Grids

In [15]:

knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

lr_param_grid = {
    'max_iter': [1000, 5000, 10000],  
    'C': [0.01, 0.1, 1, 10]           # Regularization strength
}

rf_param_grid = {
    'n_estimators' :  [50, 100, 150, 200],
    'max_depth': [8, 12, 16, 20], 
    'criterion': ["gini", "entropy", "log_loss"]
}

param_grids = {
    "KNN" : knn_param_grid,
    "Logistic Regression" : lr_param_grid,
    "RandomForest" : rf_param_grid
}

### 3. Define Multi Models

In [16]:
models = {
    "KNN": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(),
    "RandomForest": RandomForestClassifier()
}

### 4. Configure K-folds

In [17]:
cv = StratifiedKFold(
    n_splits=6,
    random_state=42,
    shuffle=True
)

### 5. Model Training

In [18]:
grid_search_results = {}

for model_name, model in models.items():
    print(f"\n--- Tuning {model_name} ---")

    param_grid = param_grids[model_name]

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='f1',
        verbose=1, return_train_score=False
    )

    print(f"Fitting gridsearchCV for {model_name}")

    grid_search.fit(X_train, y_train)

    grid_search_results[model_name] = grid_search

    print(f"{model_name} gridSearchCV completed...")
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Best CV Score: {grid_search.best_score_}")


--- Tuning KNN ---
Fitting gridsearchCV for KNN
Fitting 6 folds for each of 20 candidates, totalling 120 fits
KNN gridSearchCV completed...
Best Parameters: {'metric': 'manhattan', 'n_neighbors': 3, 'weights': 'uniform'}
Best CV Score: 0.8432734127354994

--- Tuning Logistic Regression ---
Fitting gridsearchCV for Logistic Regression
Fitting 6 folds for each of 12 candidates, totalling 72 fits
Logistic Regression gridSearchCV completed...
Best Parameters: {'C': 10, 'max_iter': 1000}
Best CV Score: 0.8338252860022927

--- Tuning RandomForest ---
Fitting gridsearchCV for RandomForest
Fitting 6 folds for each of 48 candidates, totalling 288 fits
RandomForest gridSearchCV completed...
Best Parameters: {'criterion': 'gini', 'max_depth': 20, 'n_estimators': 150}
Best CV Score: 0.8554551170770245
